In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import glob
import json
import os
import random
import shutil
from collections import Counter

import numpy as np
import yaml
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from PIL import Image
from ultralytics.utils.ops import ltwh2xyxy, xyxy2xywhn

In [ ]:
# ディレクトリの定義

generated_dir = os.path.join("data", "generated")
prepared_dir = os.path.join("data", "prepared")
raw_dir = os.path.join("data", "raw")
splited_dir = os.path.join("data", "splited")
segmented_dir = os.path.join("data", "segmented")

In [ ]:
# フォルダの切り出し

with open(os.path.join(prepared_dir, "data.yaml")) as f:
    data = yaml.safe_load(f)

names = data["names"]
categories = sorted(names.keys())

labels = sorted(
    glob.glob(os.path.join(prepared_dir, "labels", "train", "T*.txt")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

images = sorted(
    glob.glob(os.path.join(prepared_dir, "images", "train", "T*.jpg")),
    key=lambda p: int(os.path.basename(p)[1:-4]),
)

ids = [os.path.basename(label)[:-4] for label in labels]

data = np.zeros((len(ids), len(categories)), dtype=np.float32)

for i, label in enumerate(labels):
    counter = Counter()

    with open(label) as f:
        lines = f.readlines()

    for line in lines:
        counter[int(line.split(" ", 1)[0])] += 1

    for k, v in counter.items():
        data[i, k] = v

random.seed(0)
n_splits = 5

y = (data > 0).astype(int)
mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=20)
kfolds = list(mskf.split(ids, y))

folds = [f"split_{n}" for n in range(1, n_splits + 1)]

for split in folds:
    split_dir = os.path.join(splited_dir, split)

    os.makedirs(os.path.join(split_dir, "train", "images"), exist_ok=True)
    os.makedirs(os.path.join(split_dir, "train", "labels"), exist_ok=True)
    os.makedirs(os.path.join(split_dir, "val", "images"), exist_ok=True)
    os.makedirs(os.path.join(split_dir, "val", "labels"), exist_ok=True)

    with open(os.path.join(split_dir, "data.yaml"), "w") as f:
        yaml.safe_dump(
            {
                "path": split_dir,
                "train": os.path.join("train", "images"),
                "val": os.path.join("val", "images"),
                "names": names,
            },
            f,
            sort_keys=False,
        )

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    split_dir = os.path.join(splited_dir, folds[n - 1])

    for i in train_indices:
        file_name = os.path.basename(images[i])

        src = os.path.join(prepared_dir, "images", "train", file_name)
        dst = os.path.join(split_dir, "train", "images", file_name)

        shutil.copy2(src, dst)

        file_name = os.path.basename(labels[i])

        src = os.path.join(prepared_dir, "labels", "train", file_name)
        dst = os.path.join(split_dir, "train", "labels", file_name)

        shutil.copy2(src, dst)

    for i in val_indices:
        file_name = os.path.basename(images[i])

        src = os.path.join(prepared_dir, "images", "train", file_name)
        dst = os.path.join(split_dir, "val", "images", file_name)

        shutil.copy2(src, dst)

        file_name = os.path.basename(labels[i])

        src = os.path.join(prepared_dir, "labels", "train", file_name)
        dst = os.path.join(split_dir, "val", "labels", file_name)

        shutil.copy2(src, dst)

    annotation_path = os.path.join(raw_dir, "annotations", "train.json")
    with open(annotation_path, "r") as f:
        data = json.load(f)

    annotations = {ann["id"]: ann for ann in data["annotations"]}

    image_ids = {int(os.path.basename(images[i])[1:-4]) for i in train_indices}

    foregrounds = sorted(
        glob.glob(os.path.join(segmented_dir, "positive", "*.png")),
        key=lambda p: int(os.path.basename(p)[1:-4]),
    )
    foregrounds = [
        f for f in foregrounds if annotations[int(os.path.basename(f)[1:-4])]["image_id"] in image_ids
    ]

    backgrounds = sorted(
        glob.glob(os.path.join(generated_dir, "backgrounds", "*.jpg")),
        key=lambda p: int(os.path.basename(p)[1:-4]),
    )

    margin = 12

    def is_oversize(img1: Image.Image, img2: Image.Image):
        return (
            img2.width + margin * 2 > img1.width
            or img2.height + margin * 2 > img1.height
        )

    def is_overlap(box1: tuple, box2: tuple):
        return not (
            box1[2] < box2[0]
            or box1[0] > box2[2]
            or box1[3] < box2[1]
            or box1[1] > box2[3]
        )

    probs = {1: 0.15, 2: 0.15, 3: 0.25, 4: 0.25, 5: 0.20}
    population, weights = zip(*sorted(probs.items()))

    def sample():
        return random.choices(population, weights, k=1)[0]

    count = 1

    for background in backgrounds:
        bg = Image.open(background).convert("RGBA")

        boxes = []
        anns = []

        for _ in range(sample()):
            foreground = random.choice(foregrounds)
            fg = Image.open(foreground).convert("RGBA")

            if is_oversize(bg, fg):
                scale = min(
                    bg.width / (fg.width + margin * 2),
                    bg.height / (fg.height + margin * 2),
                )
                scale = scale * random.uniform(0.6, 0.8)
                fg = fg.resize(
                    (int(fg.width * scale), int(fg.height * scale)), Image.LANCZOS
                )

            for _ in range(100):
                x = random.randint(0, bg.width - fg.width - margin * 2) + margin
                y = random.randint(0, bg.height - fg.height - margin * 2) + margin
                box = (x, y, x + fg.width, y + fg.height)

                if all(not is_overlap(box, b) for b in boxes):
                    bg.alpha_composite(fg, (x, y))
                    boxes.append(box)

                    file_name = os.path.basename(foreground)
                    id = int(file_name[1:-4])
                    category_id = annotations[id]["category_id"]

                    anns.append(
                        {
                            "category_id": category_id,
                            "bbox": [x, y, fg.width, fg.height],
                        }
                    )
                    
                    break

        assert len(anns) > 0

        image = bg.convert("RGB")
        save_path = os.path.join(split_dir, "train", "images", f"A{count}.jpg")
        image.save(save_path)

        save_path = os.path.join(split_dir, "train", "labels", f"A{count}.txt")
        with open(save_path, "w") as f:
            for ann in anns:
                class_id = ann["category_id"] - 1
                xyxy = ltwh2xyxy(np.array([ann["bbox"]], dtype=np.float32))
                xywh = xyxy2xywhn(xyxy, w=image.width, h=image.height, clip=True)
                x, y, w, h = xywh[0]
                f.write(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

        count += 1